<a href="https://colab.research.google.com/github/dnysecure/TFE-Codigo-Retinopatia-IA/blob/main/tfe_codigo_donnyrodriguez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TFE: Evaluación de modelos de Deep Learning (CNN, U-Net y Transformers) para la detección temprana de la enfermedad retiniana Retinopatía diabética.
## CNN (ResNet18) · U-Net · Vision Transformer (ViT)
### Dataset: Messidor-2 · Universidad Internacional de La Rioja
---
### Dataset Messidor Link: https://www.adcis.net/en/third-party/messidor/
Se requiere subir las imagenes al drive de google en las rutas establecidas en el codigo.

> **Autor:** Donny Alexander Rodríguez Cáceres  
> **Director:** Carlos Manuel Moreno Negrin  
> **Entorno:** Google Colab + GPU T4  

| Fase | Descripción |
|------|-------------|
|  Fase 1 | Carga y exploración del dataset |
|  Fase 2 | Preprocesamiento e ingeniería de datos |
|  Fase 3 | Entrenamiento de modelos (CNN · U-Net · ViT) |
|  Fase 4 | Inferencia interactiva y comparación |
|  Fase 5 | Análisis comparativo final |


## Instalación de Dependencias

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Instalación de librerías
import subprocess, sys

packages = [
    "segmentation-models-pytorch",
    "timm",
    "scikit-learn",
    "tqdm",
    "ipywidgets",
    "opencv-python-headless",
    "psutil",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

print(" Todas las dependencias instaladas correctamente.")


##  Importaciones y Configuración Global

In [ ]:
import os, json, time, threading, gc, warnings, shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image, ImageOps
import cv2
from tqdm.notebook import tqdm
import psutil

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

import timm
import segmentation_models_pytorch as smp

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, precision_score, recall_score,
    f1_score, accuracy_score, average_precision_score
)
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True

# ╔══════════════════════════════════════════════════════╗
# ║              CONFIGURACIÓN CENTRAL                   ║
# ╚══════════════════════════════════════════════════════╝
CONFIG = {
    # Rutas (ajustar si cambian)
    'base_path'      : '/content/drive/MyDrive/Unir/TFM/dataset/messidor2/',
    'img_dir'        : '/content/drive/MyDrive/Unir/TFM/dataset/messidor2/imagenes',  # Carpeta de imágenes
    'csv_path'       : '/content/drive/MyDrive/Unir/TFM/dataset/messidor2/messidor_data.csv',
    'output_path'    : '/content/drive/MyDrive/Unir/TFM/results/',
    'processed_path' : '/content/drive/MyDrive/Unir/TFM/processed/',
    'models_path'    : '/content/drive/MyDrive/Unir/TFM/models/',
    # Imagen
    'image_size'     : 224,
    # Entrenamiento
    'batch_size'     : 16,
    'num_epochs'     : 30,
    'lr'             : 1e-4,
    'weight_decay'   : 1e-5,
    'patience'       : 7,           # Early stopping
    # Splits
    'test_size'      : 0.15,
    'val_size'       : 0.15,
    'seed'           : 42,
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CONFIG['device'] = str(DEVICE)

# Etiquetas clínicas
GRADE_LABELS = {
    0: 'Sin RD (Grado 0)',
    1: 'RD Leve (Grado 1)',
    2: 'RD Moderada (Grado 2)',
    3: 'RD Severa (Grado 3)',
    4: 'RD Proliferativa (Grado 4)'
}
GRADE_COLORS = ['#27ae60', '#f39c12', '#e67e22', '#e74c3c', '#8e44ad']
BINARY_LABELS = {0: 'Sin RD (Negativo)', 1: 'Con RD (Positivo)'}

# Crear directorios de salida
for p in [CONFIG['output_path'], CONFIG['processed_path'], CONFIG['models_path']]:
    Path(p).mkdir(parents=True, exist_ok=True)

# Info del entorno
print("=" * 55)
print(f"    Dispositivo        : {DEVICE}")
if torch.cuda.is_available():
    print(f"    GPU               : {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"    VRAM total        : {mem:.1f} GB")
print(f"    PyTorch           : {torch.__version__}")
print(f"    CPU cores         : {psutil.cpu_count()}")
ram = psutil.virtual_memory().total / 1024**3
print(f"    RAM total         : {ram:.1f} GB")
print("=" * 55)


---
#  FASE 1: Carga y Exploración del Dataset
### Dataset Messidor-2 · 1744 imágenes de retinografía · 5 grados ICDR


In [ ]:
# ── Carga del CSV ────────────────────────────────────────────────────────────
csv_path = CONFIG['csv_path']

try:
    df = pd.read_csv(csv_path)
    # Normalizar nombre de columnas por si varía entre versiones del dataset
    col_map = {}
    for c in df.columns:
        cl = c.lower().strip()
        if 'id' in cl or 'code' in cl or 'image' in cl:
            col_map[c] = 'id_code'
        elif 'diagnosis' in cl or 'dr_grade' in cl or 'grade' in cl:
            col_map[c] = 'diagnosis'
        elif 'dme' in cl:
            col_map[c] = 'adjudicated_dme'
        elif 'gradable' in cl:
            col_map[c] = 'adjudicated_gradable'
    df = df.rename(columns=col_map)

    # Filtrar sólo imágenes graduables
    df_gradable = df[df.get('adjudicated_gradable', pd.Series([1]*len(df))) == 1].copy()
    df_gradable = df_gradable.dropna(subset=['diagnosis']).reset_index(drop=True)
    df_gradable['diagnosis'] = df_gradable['diagnosis'].astype(int)

    # Etiqueta binaria: 0 = Sin RD, 1 = Con RD
    df_gradable['binary_label'] = (df_gradable['diagnosis'] > 0).astype(int)

    print(f" CSV cargado desde: {csv_path}")
    print(f"   Total imágenes en CSV         : {len(df)}")
    print(f"   Imágenes graduables           : {len(df_gradable)}")
    print()
    print(df_gradable.head(8))
except FileNotFoundError:
    raise FileNotFoundError(
        f"❌ CSV no encontrado en: {csv_path}\n"
        "   Ajusta CONFIG['csv_path'] con la ruta correcta."
    )


In [ ]:
# ── Localizar carpeta de imágenes ────────────────────────────────────────────
IMG_DIR = Path(CONFIG['img_dir'])

if not IMG_DIR.exists():
    raise FileNotFoundError(
        f" No se encontró la carpeta de imágenes en: {IMG_DIR}\n"
        "   Verifica que CONFIG['img_dir'] apunta a la ruta correcta."
    )

imgs_found = list(IMG_DIR.glob('*.png')) + list(IMG_DIR.glob('*.jpg')) + list(IMG_DIR.glob('*.jpeg'))
if len(imgs_found) == 0:
    raise FileNotFoundError(
        f" La carpeta existe pero no contiene imágenes (.png/.jpg/.jpeg): {IMG_DIR}"
    )

CONFIG['img_dir'] = str(IMG_DIR)

# Cruzar CSV con imágenes disponibles
available = {p.name for p in imgs_found}
df_gradable['img_exists'] = df_gradable['id_code'].apply(lambda x: x in available)
df_final = df_gradable[df_gradable['img_exists']].reset_index(drop=True)

print(f" Carpeta de imágenes: {IMG_DIR}")
print(f"   Imágenes en disco                : {len(available)}")
print(f"   Imágenes con CSV + imagen válida : {len(df_final)}")
print()
print("Distribución por grado:")
for g, cnt in df_final['diagnosis'].value_counts().sort_index().items():
    bar = '█' * (cnt // 20)
    print(f"  Grado {g} ({GRADE_LABELS[g][:20]:<20}): {cnt:>4}  {bar}")


In [ ]:
# ── Visualización 1: Distribución de clases ─────────────────────────────────
dist = df_final['diagnosis'].value_counts().sort_index()
dist_bin = df_final['binary_label'].value_counts().sort_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Distribución del Dataset Messidor-2', fontsize=16, fontweight='bold', y=1.01)

# — Gráfico 1: Barras por grado
bars = axes[0].bar([GRADE_LABELS[i] for i in dist.index], dist.values,
                    color=GRADE_COLORS, edgecolor='black', linewidth=0.8)
axes[0].set_title('Distribución por Grado ICDR', fontweight='bold')
axes[0].set_xlabel('Grado de Retinopatía')
axes[0].set_ylabel('Número de Imágenes')
axes[0].tick_params(axis='x', rotation=35)
for bar, val in zip(bars, dist.values):
    pct = val / len(df_final) * 100
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{val}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)

# — Gráfico 2: Torta por grado
wedges, texts, autotexts = axes[1].pie(
    dist.values,
    labels=[f'Grado {i}' for i in dist.index],
    colors=GRADE_COLORS,
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Proporción por Grado ICDR', fontweight='bold')

# — Gráfico 3: Clasificación binaria
bin_labels = [BINARY_LABELS[i] for i in dist_bin.index]
bars2 = axes[2].bar(bin_labels, dist_bin.values,
                     color=['#2ecc71', '#e74c3c'], edgecolor='black', linewidth=0.8, width=0.5)
axes[2].set_title('Clasificación Binaria\n(Objetivo Principal del TFE)', fontweight='bold')
axes[2].set_ylabel('Número de Imágenes')
for bar, val in zip(bars2, dist_bin.values):
    pct = val / len(df_final) * 100
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{val}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(CONFIG['output_path'] + 'fase1_distribucion.png', dpi=150, bbox_inches='tight')
plt.show()
print(f" Gráfico guardado en {CONFIG['output_path']}fase1_distribucion.png")


In [ ]:
# ── Visualización 2: Ejemplos por grado ──────────────────────────────
def load_image(img_name, img_dir):
    path = Path(img_dir) / img_name
    img = Image.open(path).convert('RGB')
    return img

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
fig.suptitle('Ejemplos de Retinografías — Messidor-2\n(Un ejemplo por grado ICDR)',
             fontsize=14, fontweight='bold')

for grade in range(5):
    subset = df_final[df_final['diagnosis'] == grade]
    if len(subset) == 0:
        axes[grade].text(0.5, 0.5, 'Sin muestra', ha='center', transform=axes[grade].transAxes)
        continue
    row = subset.sample(1, random_state=CONFIG['seed']).iloc[0]
    img = load_image(row['id_code'], CONFIG['img_dir'])
    axes[grade].imshow(img)
    dme = "DME: Sí" if row.get('adjudicated_dme', 0) == 1 else "DME: No"
    axes[grade].set_title(
        f"Grado {grade}\n{GRADE_LABELS[grade]}\n{dme}",
        fontsize=10, fontweight='bold',
        color=GRADE_COLORS[grade], pad=8
    )
    axes[grade].axis('off')
    # Borde de color
    for spine in axes[grade].spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(GRADE_COLORS[grade])
        spine.set_linewidth(4)

plt.tight_layout()
plt.savefig(CONFIG['output_path'] + 'fase1_ejemplos_grados.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Ejemplos guardados.")


In [ ]:
# ── Estadísticas descriptivas del dataset ────────────────────────────────────
print("="*60)
print("   RESUMEN ESTADÍSTICO — DATASET MESSIDOR-2")
print("="*60)
print(f"  Total imágenes válidas          : {len(df_final)}")
print(f"  Sin Retinopatía (Grado 0)       : {(df_final.diagnosis==0).sum()} ({(df_final.diagnosis==0).mean()*100:.1f}%)")
print(f"  Con Retinopatía (Grados 1-4)    : {(df_final.diagnosis>0).sum()} ({(df_final.diagnosis>0).mean()*100:.1f}%)")
if 'adjudicated_dme' in df_final.columns:
    print(f"  Con DME Referible               : {df_final.adjudicated_dme.sum()} ({df_final.adjudicated_dme.mean()*100:.1f}%)")
print()
print("  Distribución por grado:")
for g in range(5):
    cnt = (df_final.diagnosis == g).sum()
    pct = cnt / len(df_final) * 100
    print(f"    Grado {g} — {GRADE_LABELS[g]:<30}: {cnt:>4} ({pct:>5.1f}%)")
print("="*60)

# Guardar resumen
summary = {
    'total': len(df_final),
    'per_grade': {int(g): int(cnt) for g, cnt in df_final['diagnosis'].value_counts().sort_index().items()},
    'binary_negative': int((df_final.diagnosis==0).sum()),
    'binary_positive': int((df_final.diagnosis>0).sum()),
}
with open(CONFIG['output_path'] + 'fase1_resumen.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(" Resumen guardado en fase1_resumen.json")


---
#  FASE 2: Preprocesamiento e Ingeniería de Datos
### Normalización · CLAHE · Splits 70/15/15 · Aumento de datos


In [ ]:
# ── Función de preprocesamiento CLAHE ────────────────────────────────────────
def apply_clahe(img_pil, clip_limit=2.0, tile_grid=(8, 8)):
    """
    CLAHE (Contrast Limited Adaptive Histogram Equalization).
    Técnica estándar en imágenes de retina para mejorar el contraste
    de microaneurismas y exudados sin saturar las zonas brillantes.
    """
    img_np = np.array(img_pil)
    # Convertir a LAB para aplicar CLAHE sólo en canal de luminancia
    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l_clahe = clahe.apply(l)
    lab_clahe = cv2.merge((l_clahe, a, b))
    img_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
    return Image.fromarray(img_clahe)

def preprocess_image(img_path, target_size=224, apply_clahe_flag=True):
    """Pipeline completo: carga → recorte circular → resize → CLAHE"""
    img = Image.open(img_path).convert('RGB')
    # Resize manteniendo aspecto
    img = img.resize((target_size, target_size), Image.LANCZOS)
    if apply_clahe_flag:
        img = apply_clahe(img)
    return img

# Mostrar comparación: original vs CLAHE
sample_row = df_final[df_final['diagnosis'] == 2].sample(1, random_state=42).iloc[0]
img_orig = load_image(sample_row['id_code'], CONFIG['img_dir'])
img_orig_resized = img_orig.resize((224, 224), Image.LANCZOS)
img_clahe = apply_clahe(img_orig_resized)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Efecto del Preprocesamiento CLAHE en Retinografía (Grado 2 — RD Moderada)',
             fontsize=13, fontweight='bold')
axes[0].imshow(img_orig_resized); axes[0].set_title('Original (224×224)', fontweight='bold'); axes[0].axis('off')
axes[1].imshow(img_clahe);        axes[1].set_title('CLAHE Aplicado', fontweight='bold');     axes[1].axis('off')

# Histograma de canales
for i, (channel, color) in enumerate(zip(['R', 'G', 'B'], ['red', 'green', 'blue'])):
    orig_hist = np.histogram(np.array(img_orig_resized)[:,:,i], bins=64, range=(0,255))[0]
    clahe_hist = np.histogram(np.array(img_clahe)[:,:,i], bins=64, range=(0,255))[0]
    axes[2].plot(orig_hist, color=color, alpha=0.4, linestyle='--', label=f'{channel} orig')
    axes[2].plot(clahe_hist, color=color, alpha=0.9, linestyle='-', label=f'{channel} CLAHE')
axes[2].set_title('Histograma de Canales RGB', fontweight='bold')
axes[2].set_xlabel('Intensidad'); axes[2].set_ylabel('Frecuencia')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(CONFIG['output_path'] + 'fase2_clahe_comparacion.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── División Train / Val / Test ──────────────────────────────────────────────
SPLIT_FILE = CONFIG['processed_path'] + 'splits.json'
PREPROCESS_DONE_FILE = CONFIG['processed_path'] + 'preprocessing_done.json'

if Path(SPLIT_FILE).exists():
    print("ℹ  Splits ya existentes. Cargando desde archivo...")
    with open(SPLIT_FILE) as f:
        splits = json.load(f)
    df_train = df_final[df_final['id_code'].isin(splits['train'])].reset_index(drop=True)
    df_val   = df_final[df_final['id_code'].isin(splits['val'])].reset_index(drop=True)
    df_test  = df_final[df_final['id_code'].isin(splits['test'])].reset_index(drop=True)
    print(f" Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
else:
    # Estratificar por etiqueta binaria para mantener proporciones
    df_tv, df_test = train_test_split(
        df_final, test_size=CONFIG['test_size'],
        stratify=df_final['binary_label'], random_state=CONFIG['seed']
    )
    val_ratio = CONFIG['val_size'] / (1 - CONFIG['test_size'])
    df_train, df_val = train_test_split(
        df_tv, test_size=val_ratio,
        stratify=df_tv['binary_label'], random_state=CONFIG['seed']
    )
    df_train = df_train.reset_index(drop=True)
    df_val   = df_val.reset_index(drop=True)
    df_test  = df_test.reset_index(drop=True)

    splits = {
        'train': df_train['id_code'].tolist(),
        'val':   df_val['id_code'].tolist(),
        'test':  df_test['id_code'].tolist(),
        'created_at': datetime.now().isoformat()
    }
    with open(SPLIT_FILE, 'w') as f:
        json.dump(splits, f, indent=2)
    print(f" Splits creados y guardados.")
    print(f"   Train: {len(df_train)} ({len(df_train)/len(df_final)*100:.1f}%)")
    print(f"   Val  : {len(df_val)}   ({len(df_val)/len(df_final)*100:.1f}%)")
    print(f"   Test : {len(df_test)}  ({len(df_test)/len(df_final)*100:.1f}%)")

# Verificar balanceo de splits
for name, subset in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    pos_rate = subset['binary_label'].mean() * 100
    print(f"  {name}: {len(subset)} imgs | RD positivo: {pos_rate:.1f}%")


In [ ]:
# ── Preprocesar y guardar imágenes ───────────────────────────────────────────
processed_base = Path(CONFIG['processed_path'])

# Crear subdirectorios por split y clase binaria
for split in ['train', 'val', 'test']:
    for cls in ['0_no_rd', '1_rd']:
        (processed_base / split / cls).mkdir(parents=True, exist_ok=True)

if Path(PREPROCESS_DONE_FILE).exists():
    print("ℹ  Preprocesamiento ya completado. Saltando...")
    with open(PREPROCESS_DONE_FILE) as f:
        prep_info = json.load(f)
    print(f"   Procesadas: {prep_info['total_procesadas']} imágenes")
    print(f"   Fecha: {prep_info['fecha']}")
else:
    print(" Preprocesando imágenes...")
    stats = {'train': {'0':0,'1':0}, 'val': {'0':0,'1':0}, 'test': {'0':0,'1':0}}
    errors = []

    for split_name, df_split in [('train', df_train), ('val', df_val), ('test', df_test)]:
        pbar = tqdm(df_split.iterrows(), total=len(df_split),
                    desc=f"  Procesando {split_name}", unit='img')
        for _, row in pbar:
            src = Path(CONFIG['img_dir']) / row['id_code']
            cls_folder = '1_rd' if row['binary_label'] == 1 else '0_no_rd'
            dst = processed_base / split_name / cls_folder / row['id_code']
            if dst.exists():
                continue
            try:
                img = preprocess_image(str(src), CONFIG['image_size'])
                img.save(str(dst), quality=95)
                stats[split_name][str(row['binary_label'])] += 1
            except Exception as e:
                errors.append({'file': row['id_code'], 'error': str(e)})

    total_proc = sum(v for s in stats.values() for v in s.values())
    prep_info = {
        'total_procesadas': total_proc,
        'errores': len(errors),
        'estadisticas': stats,
        'config': {
            'image_size': CONFIG['image_size'],
            'clahe': True,
            'splits': {'train': len(df_train), 'val': len(df_val), 'test': len(df_test)}
        },
        'fecha': datetime.now().isoformat()
    }
    with open(PREPROCESS_DONE_FILE, 'w') as f:
        json.dump(prep_info, f, indent=2)
    if errors:
        with open(CONFIG['output_path'] + 'fase2_errores.json', 'w') as f:
            json.dump(errors, f, indent=2)

    print(f"\n Preprocesamiento completado!")
    print(f"   Total procesadas : {total_proc}")
    print(f"   Errores          : {len(errors)}")


In [ ]:
# ── Visualizar estadísticas del preprocesamiento ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fase 2 — Estadísticas de Preprocesamiento', fontsize=14, fontweight='bold')

splits_names = ['Train', 'Val', 'Test']
split_sizes  = [len(df_train), len(df_val), len(df_test)]
split_pos    = [(df_s['binary_label']==1).sum() for df_s in [df_train, df_val, df_test]]
split_neg    = [(df_s['binary_label']==0).sum() for df_s in [df_train, df_val, df_test]]

x = np.arange(len(splits_names))
w = 0.35
b1 = axes[0].bar(x - w/2, split_neg, w, label='Sin RD (Negativo)', color='#2ecc71', edgecolor='black')
b2 = axes[0].bar(x + w/2, split_pos, w, label='Con RD (Positivo)', color='#e74c3c', edgecolor='black')
axes[0].set_xticks(x); axes[0].set_xticklabels(splits_names, fontsize=12)
axes[0].set_ylabel('Número de Imágenes'); axes[0].set_title('Distribución por Split', fontweight='bold')
axes[0].legend()
for bar in [*b1, *b2]:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                 str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)

# Pipeline de preprocesamiento
pipeline_steps = ['Carga\nOriginal', 'Resize\n224×224', 'CLAHE\nLAB', 'Guardado\n.png']
pipeline_icons = ['📸', '📐', '🔆', '💾']
step_times_ms  = [12, 8, 25, 15]  # tiempos aproximados por imagen

bars_p = axes[1].barh(pipeline_steps, step_times_ms,
                       color=['#3498db','#9b59b6','#e67e22','#1abc9c'], edgecolor='black')
axes[1].set_xlabel('Tiempo por imagen (ms)')
axes[1].set_title('Pipeline de Preprocesamiento\n(Tiempo por etapa)', fontweight='bold')
for bar, icon in zip(bars_p, pipeline_icons):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{icon} {bar.get_width()}ms', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(CONFIG['output_path'] + 'fase2_estadisticas.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Fase 2 completada. Imágenes listas para entrenamiento.")


---
#  FASE 3: Entrenamiento de Modelos de Deep Learning
### CNN (ResNet18) · U-Net (ResNet18 encoder) · Vision Transformer (ViT-B/16)
> Mismas condiciones: 30 épocas · LR=1e-4 · Batch=16 · GPU T4


In [ ]:
# ══ Dataset PyTorch ════════════════════════════════════════════════════════
class RetinopathyDataset(Dataset):
    """Dataset para clasificación binaria de Retinopatía Diabética."""

    TRAIN_TRANSFORMS = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    EVAL_TRANSFORMS = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    def __init__(self, df, img_dir, split='train'):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = self.TRAIN_TRANSFORMS if split == 'train' else self.EVAL_TRANSFORMS

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Intentar desde directorio procesado primero
        processed_base = Path(CONFIG['processed_path'])
        for split in ['train', 'val', 'test']:
            cls = '1_rd' if row['binary_label'] == 1 else '0_no_rd'
            p = processed_base / split / cls / row['id_code']
            if p.exists():
                img = Image.open(p).convert('RGB')
                return self.transform(img), int(row['binary_label'])
        # Fallback: imagen original
        img = preprocess_image(str(self.img_dir / row['id_code']), CONFIG['image_size'])
        return self.transform(img), int(row['binary_label'])


def get_dataloaders(df_train, df_val, df_test, img_dir):
    """Crea DataLoaders con WeightedRandomSampler para controlar desbalanceo."""
    # Pesos inversamente proporcionales a la frecuencia de clase
    class_counts = df_train['binary_label'].value_counts().sort_index().values
    weights_per_class = 1.0 / class_counts
    sample_weights = torch.tensor(
        [weights_per_class[int(l)] for l in df_train['binary_label']], dtype=torch.float
    )
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    train_ds = RetinopathyDataset(df_train, img_dir, split='train')
    val_ds   = RetinopathyDataset(df_val,   img_dir, split='val')
    test_ds  = RetinopathyDataset(df_test,  img_dir, split='test')

    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], sampler=sampler,
                              num_workers=2, pin_memory=True, prefetch_factor=2)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'], shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG['batch_size'], shuffle=False,
                              num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = get_dataloaders(df_train, df_val, df_test, CONFIG['img_dir'])
print(f" DataLoaders creados:")
print(f"   Train batches : {len(train_loader)}")
print(f"   Val batches   : {len(val_loader)}")
print(f"   Test batches  : {len(test_loader)}")


In [ ]:
# ══ Monitor de Recursos del Sistema ═══════════════════════════════════════
class ResourceMonitor:
    """
    Monitorea CPU, RAM y VRAM en un hilo paralelo al entrenamiento.
    Permite medir el coste computacional real de cada modelo.
    """
    def __init__(self, interval=2.0):
        self.interval = interval
        self.data = {'cpu': [], 'ram_gb': [], 'gpu_vram_gb': [], 't': []}
        self._running = False
        self._thread = None
        self._t0 = None

    def start(self):
        self._running = True
        self._t0 = time.time()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def stop(self):
        self._running = False
        if self._thread:
            self._thread.join(timeout=5)

    def _loop(self):
        while self._running:
            self.data['cpu'].append(psutil.cpu_percent(interval=None))
            self.data['ram_gb'].append(psutil.virtual_memory().used / 1024**3)
            if torch.cuda.is_available():
                self.data['gpu_vram_gb'].append(
                    torch.cuda.memory_allocated() / 1024**3
                )
            self.data['t'].append(time.time() - self._t0)
            time.sleep(self.interval)

    def summary(self):
        s = {
            'cpu_avg': np.mean(self.data['cpu']) if self.data['cpu'] else 0,
            'cpu_max': np.max(self.data['cpu'])  if self.data['cpu'] else 0,
            'ram_avg': np.mean(self.data['ram_gb']) if self.data['ram_gb'] else 0,
            'ram_max': np.max(self.data['ram_gb'])  if self.data['ram_gb'] else 0,
        }
        if self.data['gpu_vram_gb']:
            s['vram_avg'] = np.mean(self.data['gpu_vram_gb'])
            s['vram_max'] = np.max(self.data['gpu_vram_gb'])
        return s

print(" ResourceMonitor definido.")


In [ ]:
# ══ Funciones de Entrenamiento y Evaluación ═══════════════════════════════
def compute_metrics(y_true, y_prob, threshold=0.5):
    """
    Calcula métricas clínicas completas para clasificación binaria.
    Basado en los objetivos del TFE: Sensibilidad, Especificidad, Precisión, AUC-ROC.
    """
    y_pred = (np.array(y_prob) >= threshold).astype(int)
    y_true = np.array(y_true)

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2,2) else (0, 0, 0, 0)

    sensitivity  = tp / (tp + fn + 1e-8)   # Recall positivo (detectar enfermos)
    specificity  = tn / (tn + fp + 1e-8)   # Recall negativo (no falsos positivos)
    precision    = tp / (tp + fp + 1e-8)   # Precisión predictiva
    npv          = tn / (tn + fn + 1e-8)   # Valor Predictivo Negativo
    f1           = 2 * precision * sensitivity / (precision + sensitivity + 1e-8)
    accuracy     = (tp + tn) / (tp + tn + fp + fn + 1e-8)
    auc_roc      = roc_auc_score(y_true, y_prob)
    fpr, tpr, _  = roc_curve(y_true, y_prob)

    return {
        'accuracy'   : float(accuracy),
        'sensitivity': float(sensitivity),
        'specificity': float(specificity),
        'precision'  : float(precision),
        'npv'        : float(npv),
        'f1'         : float(f1),
        'auc_roc'    : float(auc_roc),
        'confusion_matrix': cm.tolist(),
        'roc_curve'  : {'fpr': fpr.tolist(), 'tpr': tpr.tolist()},
    }


def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()
    total_loss, correct, n = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if scaler:
            with torch.cuda.amp.autocast():
                out = model(X)
                loss = criterion(out, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        n += y.size(0)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0, 0, 0
    all_labels, all_probs = [], []
    for X, y in loader:
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
        out = model(X)
        loss = criterion(out, y)
        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        total_loss += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        n += y.size(0)
        all_labels.extend(y.cpu().numpy())
        all_probs.extend(probs)
    return total_loss / n, correct / n, all_labels, all_probs


def train_model(model, model_name, train_loader, val_loader, test_loader,
                optimizer, scheduler, criterion, num_epochs, device, save_path):
    """
    Loop de entrenamiento completo con:
    - Mixed precision (AMP)
    - Early stopping
    - Monitoreo de recursos
    - Guardado del mejor modelo
    """
    scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None
    monitor = ResourceMonitor(interval=2.0)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc':  [], 'val_acc': [],
        'epoch_time': [], 'val_auc': []
    }
    best_auc = 0.0
    patience_cnt = 0

    print(f"\n{'='*60}")
    print(f"   Entrenando: {model_name}")
    print(f"     Épocas: {num_epochs} | Dispositivo: {device}")
    print(f"{'='*60}")

    monitor.start()
    t_start = time.time()

    for epoch in range(1, num_epochs + 1):
        t_ep = time.time()

        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)
        vl_loss, vl_acc, vl_labels, vl_probs = evaluate(model, val_loader, criterion, device)

        epoch_t = time.time() - t_ep
        vl_auc = roc_auc_score(vl_labels, vl_probs)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)
        history['epoch_time'].append(epoch_t)
        history['val_auc'].append(vl_auc)

        scheduler.step(vl_loss)

        if vl_auc > best_auc:
            best_auc = vl_auc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_auc': best_auc,
                'history': history,
            }, save_path)
            patience_cnt = 0
            marker = '  Best'
        else:
            patience_cnt += 1
            marker = f' (patience {patience_cnt}/{CONFIG["patience"]})'

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Época {epoch:>3}/{num_epochs} | "
                  f"Loss T={tr_loss:.4f} V={vl_loss:.4f} | "
                  f"Acc T={tr_acc:.3f} V={vl_acc:.3f} | "
                  f"AUC={vl_auc:.4f} | {epoch_t:.1f}s{marker}")

        if patience_cnt >= CONFIG['patience']:
            print(f"\n   Early stopping en época {epoch}. Mejor AUC-Val: {best_auc:.4f}")
            break

    total_time = time.time() - t_start
    monitor.stop()

    # Evaluar en Test con el mejor modelo
    ckpt = torch.load(save_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    _, _, te_labels, te_probs = evaluate(model, test_loader, criterion, device)
    test_metrics = compute_metrics(te_labels, te_probs)

    results = {
        'model_name'  : model_name,
        'history'     : history,
        'test_metrics': test_metrics,
        'total_time_s': total_time,
        'best_auc_val': best_auc,
        'resources'   : monitor.summary(),
        'resource_ts' : monitor.data,
        'epochs_run'  : len(history['train_loss']),
    }

    print(f"\n   {model_name} completado!")
    print(f"     Tiempo total      : {total_time/60:.2f} min")
    print(f"     Mejor AUC-Val     : {best_auc:.4f}")
    print(f"     AUC-ROC Test      : {test_metrics['auc_roc']:.4f}")
    print(f"     Sensibilidad Test : {test_metrics['sensitivity']:.4f}")
    print(f"     Especificidad Test: {test_metrics['specificity']:.4f}")
    print(f"     Precisión Test    : {test_metrics['precision']:.4f}")

    return results, model


def plot_training_results(results, output_path):
    """Dashboard de resultados de entrenamiento de un modelo."""
    h = results['history']
    m = results['test_metrics']
    name = results['model_name']
    epochs = list(range(1, len(h['train_loss']) + 1))

    fig = plt.figure(figsize=(20, 12))
    fig.suptitle(f'Resultados de Entrenamiento — {name}', fontsize=16, fontweight='bold')
    gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.35)

    # 1. Loss
    ax1 = fig.add_subplot(gs[0, 0:2])
    ax1.plot(epochs, h['train_loss'], 'b-o', ms=3, label='Train Loss')
    ax1.plot(epochs, h['val_loss'],   'r-o', ms=3, label='Val Loss')
    ax1.set_title('Pérdida por Época', fontweight='bold'); ax1.legend()
    ax1.set_xlabel('Época'); ax1.set_ylabel('Loss')

    # 2. AUC-ROC por época
    ax2 = fig.add_subplot(gs[0, 2:4])
    ax2.plot(epochs, h['val_auc'], 'g-o', ms=3, label='Val AUC-ROC')
    ax2.axhline(m['auc_roc'], color='purple', linestyle='--', label=f'Test AUC={m["auc_roc"]:.3f}')
    ax2.set_title('AUC-ROC por Época', fontweight='bold'); ax2.legend()
    ax2.set_xlabel('Época'); ax2.set_ylabel('AUC-ROC')
    ax2.set_ylim([0.5, 1.01])

    # 3. Tiempo por época
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.bar(epochs, h['epoch_time'], color='#3498db', alpha=0.7)
    ax3.set_title('Tiempo por Época (s)', fontweight='bold')
    ax3.set_xlabel('Época'); ax3.set_ylabel('Segundos')

    # 4. Curva ROC (Test)
    ax4 = fig.add_subplot(gs[1, 1])
    fpr = m['roc_curve']['fpr']; tpr = m['roc_curve']['tpr']
    ax4.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {m["auc_roc"]:.3f}')
    ax4.plot([0,1],[0,1],'k--',lw=1)
    ax4.set_title('Curva ROC (Test)', fontweight='bold')
    ax4.set_xlabel('1 - Especificidad (FPR)'); ax4.set_ylabel('Sensibilidad (TPR)')
    ax4.legend(); ax4.set_xlim([-0.01,1]); ax4.set_ylim([0,1.01])

    # 5. Matriz de confusión
    ax5 = fig.add_subplot(gs[1, 2])
    cm = np.array(m['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax5,
                xticklabels=['Sin RD','Con RD'], yticklabels=['Sin RD','Con RD'])
    ax5.set_title('Matriz de Confusión (Test)', fontweight='bold')
    ax5.set_xlabel('Predicción'); ax5.set_ylabel('Real')

    # 6. Métricas resumen
    ax6 = fig.add_subplot(gs[1, 3])
    metric_names = ['Sensibilidad','Especificidad','Precisión','F1-Score','AUC-ROC','Exactitud']
    metric_vals  = [m['sensitivity'], m['specificity'], m['precision'],
                    m['f1'], m['auc_roc'], m['accuracy']]
    colors_m = ['#e74c3c','#2ecc71','#3498db','#f39c12','#9b59b6','#1abc9c']
    bars = ax6.barh(metric_names, metric_vals, color=colors_m, edgecolor='black')
    ax6.set_xlim([0, 1.1]); ax6.set_title('Métricas Clínicas (Test)', fontweight='bold')
    for bar, val in zip(bars, metric_vals):
        ax6.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9, fontweight='bold')

    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f" Dashboard guardado: {output_path}")


def plot_resources(results, output_path):
    """Gráfico de uso de recursos durante el entrenamiento."""
    ts_data = results['resource_ts']
    if not ts_data['t']:
        print("Sin datos de recursos disponibles.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f"Uso de Recursos — {results['model_name']}", fontsize=13, fontweight='bold')
    t = ts_data['t']

    axes[0].plot(t, ts_data['cpu'], color='#3498db'); axes[0].fill_between(t, ts_data['cpu'], alpha=0.3, color='#3498db')
    axes[0].set_title('CPU (%)'); axes[0].set_xlabel('Tiempo (s)'); axes[0].set_ylabel('%'); axes[0].set_ylim([0,100])

    axes[1].plot(t, ts_data['ram_gb'], color='#e67e22'); axes[1].fill_between(t, ts_data['ram_gb'], alpha=0.3, color='#e67e22')
    axes[1].set_title('RAM (GB)'); axes[1].set_xlabel('Tiempo (s)'); axes[1].set_ylabel('GB')

    if ts_data['gpu_vram_gb']:
        axes[2].plot(t[:len(ts_data['gpu_vram_gb'])], ts_data['gpu_vram_gb'], color='#8e44ad')
        axes[2].fill_between(t[:len(ts_data['gpu_vram_gb'])], ts_data['gpu_vram_gb'], alpha=0.3, color='#8e44ad')
        axes[2].set_title('VRAM GPU (GB)'); axes[2].set_xlabel('Tiempo (s)'); axes[2].set_ylabel('GB')
    else:
        axes[2].text(0.5, 0.5, 'GPU no disponible', ha='center', transform=axes[2].transAxes)

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()


print(" Funciones de entrenamiento y evaluación definidas.")


---
##  FASE 3a — Modelo 1: CNN con backbone ResNet18
> Arquitectura: ResNet18 pre-entrenado (ImageNet) + Fine-tuning completo  
> Tarea: Clasificación binaria (Sin RD vs Con RD)


In [ ]:
# ── Definición del modelo CNN ─────────────────────────────────────────────
class CNNResNet18(nn.Module):
    """
    ResNet18 con fine-tuning completo para clasificación binaria de RD.
    - Backbone preentrenado en ImageNet para transfer learning
    - Cabeza clasificadora con Dropout para regularización
    - Capa de BatchNorm adicional para adaptación al dominio médico
    """
    def __init__(self, num_classes=2, dropout_rate=0.5):
        super().__init__()
        backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        # Extraer todas las capas excepto la FC final
        self.features = nn.Sequential(*list(backbone.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


# Instanciar
cnn_model = CNNResNet18(num_classes=2, dropout_rate=0.4).to(DEVICE)
total_params = sum(p.numel() for p in cnn_model.parameters())
trainable   = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"  Parámetros totales     : {total_params:,}")
print(f"  Parámetros entrenables : {trainable:,}")

# Pérdida con pesos de clase para desbalanceo
pos_weight = (df_train['binary_label'] == 0).sum() / (df_train['binary_label'] == 1).sum()
class_weights = torch.tensor([1.0, pos_weight], dtype=torch.float).to(DEVICE)
criterion_cnn = nn.CrossEntropyLoss(weight=class_weights)

# Optimizador y scheduler
optimizer_cnn = optim.AdamW(cnn_model.parameters(),
                             lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler_cnn = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_cnn, mode='min', factor=0.5, patience=3)

print(" CNN (ResNet18) listo para entrenamiento.")


In [ ]:
# ── Entrenamiento CNN ────────────────────────────────────────────────────────
CNN_MODEL_PATH   = CONFIG['models_path'] + 'cnn_resnet18_best.pth'
CNN_RESULTS_PATH = CONFIG['output_path'] + 'cnn_results.json'

if Path(CNN_RESULTS_PATH).exists():
    print("ℹ  Resultados CNN ya existen. Cargando...")
    with open(CNN_RESULTS_PATH) as f:
        cnn_results = json.load(f)
    # Cargar pesos para uso posterior
    ckpt = torch.load(CNN_MODEL_PATH, map_location=DEVICE, weights_only=False)
    cnn_model.load_state_dict(ckpt['model_state_dict'])
    print(f"   AUC-ROC Test: {cnn_results['test_metrics']['auc_roc']:.4f}")
else:
    cnn_results, cnn_model = train_model(
        model=cnn_model,
        model_name='CNN (ResNet18)',
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        optimizer=optimizer_cnn,
        scheduler=scheduler_cnn,
        criterion=criterion_cnn,
        num_epochs=CONFIG['num_epochs'],
        device=DEVICE,
        save_path=CNN_MODEL_PATH,
    )
    with open(CNN_RESULTS_PATH, 'w') as f:
        json.dump(cnn_results, f, indent=2)
    print(f" Resultados CNN guardados.")


In [ ]:
# ── Dashboard CNN ────────────────────────────────────────────────────────────
plot_training_results(cnn_results, CONFIG['output_path'] + 'fase3a_cnn_dashboard.png')
plot_resources(cnn_results, CONFIG['output_path'] + 'fase3a_cnn_recursos.png')

# Tabla resumen
m = cnn_results['test_metrics']
r = cnn_results['resources']
print("\n RESUMEN CNN (ResNet18) — Conjunto de Test")
print("─" * 50)
print(f"  AUC-ROC       : {m['auc_roc']:.4f}")
print(f"  Sensibilidad  : {m['sensitivity']:.4f}  (Recall enfermos)")
print(f"  Especificidad : {m['specificity']:.4f}  (Recall sanos)")
print(f"  Precisión     : {m['precision']:.4f}")
print(f"  F1-Score      : {m['f1']:.4f}")
print(f"  Exactitud     : {m['accuracy']:.4f}")
print(f"  Épocas        : {cnn_results['epochs_run']}")
print(f"  Tiempo total  : {cnn_results['total_time_s']/60:.2f} min")
print(f"  CPU avg/max   : {r.get('cpu_avg',0):.1f}% / {r.get('cpu_max',0):.1f}%")
print(f"  RAM avg/max   : {r.get('ram_avg',0):.2f} / {r.get('ram_max',0):.2f} GB")
if 'vram_avg' in r:
    print(f"  VRAM avg/max  : {r['vram_avg']:.2f} / {r['vram_max']:.2f} GB")
print("─" * 50)


---
##  FASE 3b — Modelo 2: U-Net con encoder ResNet18
> Arquitectura: U-Net (segmentation_models_pytorch) + cabeza de clasificación global  
> Encoder pre-entrenado en ImageNet, decoder adaptado para clasificación binaria


In [ ]:
# ── Definición del modelo U-Net ──────────────────────────────────────────────
class UNetClassifier(nn.Module):
    """
    U-Net adaptada para clasificación binaria de imágenes retinianas.
    El encoder ResNet18 extrae características jerárquicas.
    El decoder reconstruye el mapa de características a resolución original.
    Una cabeza global convierte el mapa del decoder en probabilidades de clase.
    Esta arquitectura aprovecha la capacidad de U-Net para detectar lesiones
    pequeñas (microaneurismas, exudados) relevantes en RD.
    """
    def __init__(self, encoder_name='resnet18', num_classes=2, dropout=0.4):
        super().__init__()
        self.unet = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights='imagenet',
            in_channels=3,
            classes=32,             # Features del decoder (no clases finales)
            activation=None,
        )
        # Cabeza de clasificación global
        self.global_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(32 * 4 * 4),
            nn.Dropout(dropout),
            nn.Linear(32 * 4 * 4, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        feat_map = self.unet(x)          # [B, 32, H, W]
        pooled   = self.global_pool(feat_map)  # [B, 32, 4, 4]
        return self.classifier(pooled)


unet_model = UNetClassifier(encoder_name='resnet18', num_classes=2).to(DEVICE)
total_params = sum(p.numel() for p in unet_model.parameters())
trainable   = sum(p.numel() for p in unet_model.parameters() if p.requires_grad)
print(f"  Parámetros totales     : {total_params:,}")
print(f"  Parámetros entrenables : {trainable:,}")

criterion_unet = nn.CrossEntropyLoss(weight=class_weights)
optimizer_unet = optim.AdamW(unet_model.parameters(),
                              lr=CONFIG['lr'] * 0.5, weight_decay=CONFIG['weight_decay'])
scheduler_unet = optim.lr_scheduler.ReduceLROnPlateau(optimizer_unet, mode='min',
                                                       factor=0.5, patience=3)
print(" U-Net (ResNet18 encoder) listo para entrenamiento.")


In [ ]:
# ── Entrenamiento U-Net ──────────────────────────────────────────────────────
UNET_MODEL_PATH   = CONFIG['models_path'] + 'unet_resnet18_best.pth'
UNET_RESULTS_PATH = CONFIG['output_path'] + 'unet_results.json'

if Path(UNET_RESULTS_PATH).exists():
    print("ℹ  Resultados U-Net ya existen. Cargando...")
    with open(UNET_RESULTS_PATH) as f:
        unet_results = json.load(f)
    ckpt = torch.load(UNET_MODEL_PATH, map_location=DEVICE, weights_only=False)
    unet_model.load_state_dict(ckpt['model_state_dict'])
    print(f"   AUC-ROC Test: {unet_results['test_metrics']['auc_roc']:.4f}")
else:
    unet_results, unet_model = train_model(
        model=unet_model,
        model_name='U-Net (ResNet18 encoder)',
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        optimizer=optimizer_unet,
        scheduler=scheduler_unet,
        criterion=criterion_unet,
        num_epochs=CONFIG['num_epochs'],
        device=DEVICE,
        save_path=UNET_MODEL_PATH,
    )
    with open(UNET_RESULTS_PATH, 'w') as f:
        json.dump(unet_results, f, indent=2)
    print(" Resultados U-Net guardados.")


In [ ]:
# ── Dashboard U-Net ──────────────────────────────────────────────────────────
plot_training_results(unet_results, CONFIG['output_path'] + 'fase3b_unet_dashboard.png')
plot_resources(unet_results, CONFIG['output_path'] + 'fase3b_unet_recursos.png')

m = unet_results['test_metrics']
r = unet_results['resources']
print("\n RESUMEN U-Net — Conjunto de Test")
print("─" * 50)
print(f"  AUC-ROC       : {m['auc_roc']:.4f}")
print(f"  Sensibilidad  : {m['sensitivity']:.4f}")
print(f"  Especificidad : {m['specificity']:.4f}")
print(f"  Precisión     : {m['precision']:.4f}")
print(f"  F1-Score      : {m['f1']:.4f}")
print(f"  Exactitud     : {m['accuracy']:.4f}")
print(f"  Épocas        : {unet_results['epochs_run']}")
print(f"  Tiempo total  : {unet_results['total_time_s']/60:.2f} min")
print(f"  CPU avg/max   : {r.get('cpu_avg',0):.1f}% / {r.get('cpu_max',0):.1f}%")
if 'vram_avg' in r:
    print(f"  VRAM avg/max  : {r['vram_avg']:.2f} / {r['vram_max']:.2f} GB")
print("─" * 50)


---
##  FASE 3c — Modelo 3: Vision Transformer (ViT-B/16)
> Arquitectura: ViT-B/16 pre-entrenado (ImageNet-21k) + Fine-tuning  
> Mecanismo de auto-atención global para capturar dependencias de largo alcance


In [ ]:
# ── Definición del modelo ViT ────────────────────────────────────────────────
class VisionTransformerClassifier(nn.Module):
    """
    Vision Transformer (ViT-B/16) para clasificación de RD.
    La tokenización por patches (16×16) permite modelar relaciones
    globales entre lesiones retinianas distribuidas en la imagen,
    lo que supera la limitación del campo receptivo local de las CNN.
    """
    def __init__(self, num_classes=2, dropout=0.4, pretrained=True):
        super().__init__()
        model_name = 'vit_base_patch16_224'
        self.vit = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,          # Quitar cabeza original
            drop_rate=dropout,
            attn_drop_rate=0.1,
        )
        embed_dim = self.vit.embed_dim  # 768 para ViT-B
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        features = self.vit(x)  # [B, 768]
        return self.head(features)


vit_model = VisionTransformerClassifier(num_classes=2, pretrained=True).to(DEVICE)
total_params = sum(p.numel() for p in vit_model.parameters())
trainable   = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)
print(f"  Parámetros totales     : {total_params:,}")
print(f"  Parámetros entrenables : {trainable:,}")

# ViT necesita LR más pequeño con warm-up
criterion_vit = nn.CrossEntropyLoss(weight=class_weights)
optimizer_vit = optim.AdamW(vit_model.parameters(),
                             lr=CONFIG['lr'] * 0.3, weight_decay=1e-4,
                             betas=(0.9, 0.999))
scheduler_vit = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_vit, T_max=CONFIG['num_epochs'], eta_min=1e-6
)
print(" Vision Transformer (ViT-B/16) listo para entrenamiento.")


In [ ]:
# ── Entrenamiento ViT ────────────────────────────────────────────────────────
VIT_MODEL_PATH   = CONFIG['models_path'] + 'vit_b16_best.pth'
VIT_RESULTS_PATH = CONFIG['output_path'] + 'vit_results.json'

if Path(VIT_RESULTS_PATH).exists():
    print("ℹ  Resultados ViT ya existen. Cargando...")
    with open(VIT_RESULTS_PATH) as f:
        vit_results = json.load(f)
    ckpt = torch.load(VIT_MODEL_PATH, map_location=DEVICE, weights_only=False)
    vit_model.load_state_dict(ckpt['model_state_dict'])
    print(f"   AUC-ROC Test: {vit_results['test_metrics']['auc_roc']:.4f}")
else:
    vit_results, vit_model = train_model(
        model=vit_model,
        model_name='Vision Transformer (ViT-B/16)',
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        optimizer=optimizer_vit,
        scheduler=scheduler_vit,
        criterion=criterion_vit,
        num_epochs=CONFIG['num_epochs'],
        device=DEVICE,
        save_path=VIT_MODEL_PATH,
    )
    with open(VIT_RESULTS_PATH, 'w') as f:
        json.dump(vit_results, f, indent=2)
    print(" Resultados ViT guardados.")


In [ ]:
# ── Dashboard ViT ────────────────────────────────────────────────────────────
plot_training_results(vit_results, CONFIG['output_path'] + 'fase3c_vit_dashboard.png')
plot_resources(vit_results, CONFIG['output_path'] + 'fase3c_vit_recursos.png')

m = vit_results['test_metrics']
r = vit_results['resources']
print("\n RESUMEN ViT-B/16 — Conjunto de Test")
print("─" * 50)
print(f"  AUC-ROC       : {m['auc_roc']:.4f}")
print(f"  Sensibilidad  : {m['sensitivity']:.4f}")
print(f"  Especificidad : {m['specificity']:.4f}")
print(f"  Precisión     : {m['precision']:.4f}")
print(f"  F1-Score      : {m['f1']:.4f}")
print(f"  Exactitud     : {m['accuracy']:.4f}")
print(f"  Épocas        : {vit_results['epochs_run']}")
print(f"  Tiempo total  : {vit_results['total_time_s']/60:.2f} min")
if 'vram_avg' in r:
    print(f"  VRAM avg/max  : {r['vram_avg']:.2f} / {r['vram_max']:.2f} GB")
print("─" * 50)


---
#  FASE 4: Inferencia Interactiva sobre Nueva Imagen
### Sube una imagen de retina → los 3 modelos la analizan simultáneamente


In [ ]:
# ── Motor de Inferencia ──────────────────────────────────────────────────────
INFER_TRANSFORM = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def predict_image(model, img_pil, device):
    """
    Devuelve la probabilidad de RD y la clase predicha para una imagen PIL.
    Aplica CLAHE como parte del preprocesamiento (igual que el entrenamiento).
    """
    model.eval()
    img = apply_clahe(img_pil.resize((CONFIG['image_size'], CONFIG['image_size']), Image.LANCZOS))
    tensor = INFER_TRANSFORM(img).unsqueeze(0).to(device)
    logits = model(tensor)
    probs  = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred   = int(probs.argmax())
    return pred, float(probs[1])   # clase, prob_RD


def classify_grade(prob_rd):
    """Mapeo heurístico de probabilidad binaria a grado estimado."""
    if prob_rd < 0.15: return 0, GRADE_LABELS[0]
    if prob_rd < 0.35: return 1, GRADE_LABELS[1]
    if prob_rd < 0.60: return 2, GRADE_LABELS[2]
    if prob_rd < 0.80: return 3, GRADE_LABELS[3]
    return 4, GRADE_LABELS[4]


def run_inference_all_models(img_pil):
    """Ejecuta los 3 modelos sobre la imagen y retorna resultados."""
    models_dict = {
        'CNN (ResNet18)': cnn_model,
        'U-Net (ResNet18)': unet_model,
        'ViT-B/16': vit_model,
    }
    results_inf = {}
    for name, mdl in models_dict.items():
        pred, prob = predict_image(mdl, img_pil, DEVICE)
        grade, grade_label = classify_grade(prob)
        results_inf[name] = {
            'prediccion': pred,
            'prob_rd': prob,
            'grado_estimado': grade,
            'etiqueta_grado': grade_label,
        }
    return results_inf


print(" Motor de inferencia listo.")


In [ ]:
# ── Widget de Carga y Análisis de Imagen ────────────────────────────────────
from google.colab import files as colab_files

def analyze_uploaded_image():
    """
    Interfaz interactiva para subir una imagen retiniana y analizarla
    con los 3 modelos entrenados.
    """
    print(" Selecciona una imagen de retina (.png, .jpg, .jpeg):")
    print("   (Usa el botón que aparece a continuación)")

    uploaded = colab_files.upload()
    if not uploaded:
        print("❌ No se subió ningún archivo.")
        return

    fname = list(uploaded.keys())[0]
    img_bytes = uploaded[fname]

    from io import BytesIO
    img_pil = Image.open(BytesIO(img_bytes)).convert('RGB')

    print(f"\n Imagen cargada: {fname} ({img_pil.size[0]}×{img_pil.size[1]} px)")
    print(" Analizando con los 3 modelos...")

    results_inf = run_inference_all_models(img_pil)

    # ── Visualización ────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(20, 10))
    fig.suptitle(f'Análisis de Retinopatía Diabética — {fname}',
                 fontsize=15, fontweight='bold')
    gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.35)

    # Imagen original (col 0, span 2 rows)
    ax_img = fig.add_subplot(gs[:, 0])
    ax_img.imshow(img_pil)
    ax_img.set_title('Imagen Analizada', fontweight='bold', fontsize=12)
    ax_img.axis('off')

    model_names = list(results_inf.keys())
    colors_res  = []
    for i, (mname, res) in enumerate(results_inf.items()):
        ax = fig.add_subplot(gs[0, i+1])
        prob = res['prob_rd']
        grade = res['grado_estimado']
        color = GRADE_COLORS[grade]
        colors_res.append(color)

        # Gauge de probabilidad
        theta = np.linspace(0, np.pi, 200)
        ax.plot(np.cos(theta), np.sin(theta), 'k-', lw=2)
        fill_end = int(prob * 200)
        ax.fill_between(np.cos(theta[:fill_end]), 0, np.sin(theta[:fill_end]),
                         alpha=0.7, color=color)
        ax.text(0, -0.2, f'{prob*100:.1f}%', ha='center', fontsize=18, fontweight='bold', color=color)
        ax.set_title(f'{mname}\n{res["etiqueta_grado"]}',
                     fontweight='bold', fontsize=10, color=color)
        ax.set_xlim(-1.2, 1.2); ax.set_ylim(-0.4, 1.2)
        ax.axis('off')

    # Comparativa de probabilidades
    ax_comp = fig.add_subplot(gs[1, 1:4])
    probs = [results_inf[n]['prob_rd'] * 100 for n in model_names]
    bars = ax_comp.bar(model_names, probs, color=colors_res, edgecolor='black', linewidth=1.2)
    ax_comp.axhline(50, color='red', linestyle='--', lw=1.5, label='Umbral 50%')
    ax_comp.set_title('Probabilidad de Retinopatía Diabética por Modelo', fontweight='bold')
    ax_comp.set_ylabel('Probabilidad (%)'); ax_comp.set_ylim([0, 110])
    ax_comp.legend()
    for bar, prob in zip(bars, probs):
        ax_comp.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                     f'{prob:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

    plt.savefig(CONFIG['output_path'] + 'fase4_inferencia.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Tabla de resultados ──────────────────────────────────────────────────
    print("\n" + "═"*65)
    print("   DIAGNÓSTICO POR MODELO")
    print("═"*65)
    print(f"  {'Modelo':<25} {'Prob. RD':>10} {'Grado':>8} {'Diagnóstico':<30}")
    print("─"*65)
    decisions = []
    for mname, res in results_inf.items():
        status = ' RD POSITIVO' if res['prediccion'] == 1 else ' SIN RD'
        print(f"  {mname:<25} {res['prob_rd']*100:>9.1f}% {res['grado_estimado']:>8} {status}")
        decisions.append(res['prediccion'])
    print("═"*65)
    consensus = sum(decisions)
    if consensus >= 2:
        print(f"\n    CONSENSO (≥2 modelos): RETINOPATÍA DIABÉTICA DETECTADA")
        print(f"     → Se recomienda evaluación clínica urgente.")
    else:
        print(f"\n    CONSENSO (≥2 modelos): SIN INDICIOS DE RETINOPATÍA")
        print(f"     → Continuar con seguimiento regular.")
    print("\n    Este sistema es de apoyo diagnóstico. No reemplaza al médico.")

    return results_inf


# Ejecutar interfaz
results_fase4 = analyze_uploaded_image()


---
#  FASE 5: Comparativa Final — CNN vs U-Net vs ViT
### Análisis cuantitativo y cualitativo de los tres modelos


In [ ]:
# ── Tabla comparativa principal ──────────────────────────────────────────────
all_results = {
    'CNN (ResNet18)' : cnn_results,
    'U-Net (ResNet18)': unet_results,
    'ViT-B/16'       : vit_results,
}

rows = []
for name, res in all_results.items():
    m = res['test_metrics']
    r = res['resources']
    rows.append({
        'Modelo'            : name,
        'AUC-ROC'           : round(m['auc_roc'], 4),
        'Sensibilidad'      : round(m['sensitivity'], 4),
        'Especificidad'     : round(m['specificity'], 4),
        'Precisión'         : round(m['precision'], 4),
        'F1-Score'          : round(m['f1'], 4),
        'Exactitud'         : round(m['accuracy'], 4),
        'Épocas'            : res['epochs_run'],
        'T. Total (min)'    : round(res['total_time_s'] / 60, 2),
        'T. Época avg (s)'  : round(np.mean(res['history']['epoch_time']), 2),
        'CPU avg (%)'       : round(r.get('cpu_avg', 0), 1),
        'RAM max (GB)'      : round(r.get('ram_max', 0), 2),
        'VRAM max (GB)'     : round(r.get('vram_max', 0), 2),
        'Parámetros (M)'    : round(sum(p.numel() for p in [cnn_model, unet_model, vit_model][
            list(all_results.keys()).index(name)].parameters()) / 1e6, 2),
    })

df_compare = pd.DataFrame(rows).set_index('Modelo')

# Resaltar máximos/mínimos clínicos
print("╔" + "═"*70 + "╗")
print("║  TABLA COMPARATIVA COMPLETA — TFE Donny Rodríguez                   ║")
print("║  Dataset: Messidor-2 | Clasificación Binaria RD vs Sin RD           ║")
print("╚" + "═"*70 + "╝")
print(df_compare.T.to_string())

# Guardar CSV
df_compare.to_csv(CONFIG['output_path'] + 'fase5_comparativa_completa.csv')
print(f"\n Tabla guardada en fase5_comparativa_completa.csv")

# ── Determinar ganador por métrica ──────────────────────────────────────────
print("\n MEJOR MODELO POR MÉTRICA CLÍNICA:")
print("─" * 50)
for metric in ['AUC-ROC', 'Sensibilidad', 'Especificidad', 'Precisión', 'F1-Score']:
    best = df_compare[metric].idxmax()
    val  = df_compare[metric].max()
    print(f"  {metric:<20}: {best} ({val:.4f})")
print("─" * 50)
fastest = df_compare['T. Total (min)'].idxmin()
print(f"  {'Más rápido':<20}: {fastest} ({df_compare['T. Total (min)'].min():.2f} min)")
lightest = df_compare['Parámetros (M)'].idxmin()
print(f"  {'Menos parámetros':<20}: {lightest} ({df_compare['Parámetros (M)'].min():.2f} M)")


In [ ]:
# ── Dashboard comparativo final (gráficas) ───────────────────────────────────
fig = plt.figure(figsize=(22, 18))
fig.suptitle('FASE 5 — Comparativa Final: CNN vs U-Net vs Vision Transformer\nDataset Messidor-2 | Clasificación Binaria de Retinopatía Diabética',
             fontsize=16, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.50, wspace=0.35)

names   = list(all_results.keys())
palette = ['#2980b9', '#27ae60', '#8e44ad']

# 1 — Radar de métricas clínicas
ax1 = fig.add_subplot(gs[0, 0:2], polar=True)
metrics_radar = ['AUC-ROC', 'Sensibilidad', 'Especificidad', 'Precisión', 'F1-Score']
angles = np.linspace(0, 2*np.pi, len(metrics_radar), endpoint=False).tolist()
angles += angles[:1]
for i, (name, res) in enumerate(all_results.items()):
    m = res['test_metrics']
    vals = [m['auc_roc'], m['sensitivity'], m['specificity'], m['precision'], m['f1']]
    vals += vals[:1]
    ax1.plot(angles, vals, 'o-', lw=2, color=palette[i], label=name)
    ax1.fill(angles, vals, alpha=0.12, color=palette[i])
ax1.set_thetagrids(np.degrees(angles[:-1]), metrics_radar)
ax1.set_ylim([0, 1]); ax1.set_title('Radar — Métricas Clínicas', fontweight='bold', pad=20)
ax1.legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=9)

# 2 — Barras métricas clínicas
ax2 = fig.add_subplot(gs[0, 2:4])
x  = np.arange(len(metrics_radar))
w  = 0.25
for i, (name, res) in enumerate(all_results.items()):
    m = res['test_metrics']
    vals = [m['auc_roc'], m['sensitivity'], m['specificity'], m['precision'], m['f1']]
    ax2.bar(x + i*w, vals, w, label=name, color=palette[i], edgecolor='black', linewidth=0.6)
ax2.set_xticks(x + w); ax2.set_xticklabels(metrics_radar, rotation=20, ha='right')
ax2.set_ylabel('Valor'); ax2.set_ylim([0.5, 1.08])
ax2.set_title('Comparativa de Métricas Clínicas', fontweight='bold')
ax2.legend(fontsize=9); ax2.axhline(0.9, color='red', linestyle='--', lw=1, alpha=0.6, label='AUC 0.90')

# 3 — Curvas ROC superpuestas
ax3 = fig.add_subplot(gs[1, 0:2])
for i, (name, res) in enumerate(all_results.items()):
    fpr = res['test_metrics']['roc_curve']['fpr']
    tpr = res['test_metrics']['roc_curve']['tpr']
    auc = res['test_metrics']['auc_roc']
    ax3.plot(fpr, tpr, lw=2.5, color=palette[i], label=f'{name} (AUC={auc:.3f})')
ax3.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
ax3.fill_between([0,1],[0,1], alpha=0.05, color='gray')
ax3.set_title('Curvas ROC Comparativas', fontweight='bold')
ax3.set_xlabel('1 - Especificidad (FPR)'); ax3.set_ylabel('Sensibilidad (TPR)')
ax3.legend(fontsize=10); ax3.set_xlim([-0.01,1]); ax3.set_ylim([0,1.01])
ax3.grid(alpha=0.3)

# 4 — Tiempo por época
ax4 = fig.add_subplot(gs[1, 2:4])
for i, (name, res) in enumerate(all_results.items()):
    times = res['history']['epoch_time']
    ep    = list(range(1, len(times)+1))
    ax4.plot(ep, times, '-o', ms=3, color=palette[i], label=name, lw=1.8)
ax4.set_title('Tiempo por Época (s)', fontweight='bold')
ax4.set_xlabel('Época'); ax4.set_ylabel('Segundos')
ax4.legend(fontsize=9); ax4.grid(alpha=0.3)

# 5 — Coste computacional
ax5 = fig.add_subplot(gs[2, 0])
metrics_cost = ['T. Total (min)', 'T. Época avg (s)']
x2 = np.arange(len(metrics_cost)); w2 = 0.25
for i, name in enumerate(names):
    vals = [df_compare.loc[name, m] for m in metrics_cost]
    ax5.bar(x2 + i*w2, vals, w2, label=name, color=palette[i], edgecolor='black')
ax5.set_xticks(x2+w2); ax5.set_xticklabels(metrics_cost, fontsize=9)
ax5.set_title('Coste Computacional\n(Tiempo)', fontweight='bold'); ax5.legend(fontsize=8)

# 6 — VRAM y RAM
ax6 = fig.add_subplot(gs[2, 1])
ram_vals  = [df_compare.loc[n, 'RAM max (GB)'] for n in names]
vram_vals = [df_compare.loc[n, 'VRAM max (GB)'] for n in names]
x3 = np.arange(len(names)); w3 = 0.35
ax6.bar(x3 - w3/2, ram_vals,  w3, label='RAM max (GB)',  color='#e67e22', edgecolor='black')
ax6.bar(x3 + w3/2, vram_vals, w3, label='VRAM max (GB)', color='#8e44ad', edgecolor='black')
ax6.set_xticks(x3); ax6.set_xticklabels([n.split('(')[0].strip() for n in names], fontsize=9)
ax6.set_title('Uso de Memoria\n(RAM y VRAM)', fontweight='bold'); ax6.legend(fontsize=8)

# 7 — Parámetros del modelo
ax7 = fig.add_subplot(gs[2, 2])
param_vals = [df_compare.loc[n, 'Parámetros (M)'] for n in names]
bars = ax7.bar([n.split('(')[0].strip() for n in names], param_vals,
               color=palette, edgecolor='black')
ax7.set_title('Número de Parámetros (M)', fontweight='bold')
for bar, val in zip(bars, param_vals):
    ax7.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
             f'{val}M', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 8 — Val Loss evolution
ax8 = fig.add_subplot(gs[2, 3])
for i, (name, res) in enumerate(all_results.items()):
    ep = list(range(1, len(res['history']['val_loss'])+1))
    ax8.plot(ep, res['history']['val_loss'], '-', color=palette[i], label=name, lw=2)
ax8.set_title('Val Loss por Época', fontweight='bold')
ax8.set_xlabel('Época'); ax8.set_ylabel('Loss')
ax8.legend(fontsize=8); ax8.grid(alpha=0.3)

plt.savefig(CONFIG['output_path'] + 'fase5_comparativa_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Dashboard comparativo guardado.")


In [ ]:
# ── Matrices de confusión comparativas ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Matrices de Confusión Comparativas — Conjunto de Test', fontsize=14, fontweight='bold')

for ax, (name, res), color in zip(axes, all_results.items(), palette):
    cm = np.array(res['test_metrics']['confusion_matrix'])
    m  = res['test_metrics']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Sin RD','Con RD'], yticklabels=['Sin RD','Con RD'],
                linewidths=0.5, linecolor='black',
                annot_kws={"size": 14, "weight": "bold"})
    ax.set_title(f'{name}\nAUC={m["auc_roc"]:.3f} | Sens={m["sensitivity"]:.3f} | Espec={m["specificity"]:.3f}',
                 fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('Predicción'); ax.set_ylabel('Real')

plt.tight_layout()
plt.savefig(CONFIG['output_path'] + 'fase5_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Análisis cualitativo y conclusión final ──────────────────────────────────
print("╔" + "═"*70 + "╗")
print("║  ANÁLISIS COMPARATIVO FINAL — TFE UNIR 2026                        ║")
print("║  Evaluación de Modelos DL para Detección de Retinopatía Diabética  ║")
print("╚" + "═"*70 + "╝\n")

ventajas = {
    'CNN (ResNet18)': [
        ' Menor número de parámetros → más liviano',
        ' Entrenamiento más rápido',
        ' Excelente extracción de características locales',
        ' Mayor madurez y herramientas de interpretabilidad (GradCAM)',
        '  Campo receptivo limitado al tamaño del kernel',
    ],
    'U-Net (ResNet18)': [
        ' Preserva información espacial a múltiples escalas',
        ' Capacidad de detectar lesiones pequeñas (microaneurismas)',
        ' Encoder-Decoder simétrico ideal para imágenes médicas',
        '  Mayor número de parámetros que CNN pura',
        '  Pensada para segmentación; adaptación a clasificación añade complejidad',
    ],
    'ViT-B/16': [
        ' Modelado de dependencias globales (atención multi-cabeza)',
        ' Escalabilidad con mayor datos',
        ' Competitivo con CNN a igual tamaño de datos',
        '  Mayor número de parámetros (86M+)',
        '  Mayor coste computacional y de memoria',
        '  Requiere más datos o pre-entrenamiento fuerte para generalizar',
    ],
}

for name, items in ventajas.items():
    m  = all_results[name]['test_metrics']
    print(f"─ {name} {'─'*(50-len(name))}")
    print(f"  AUC-ROC={m['auc_roc']:.3f} | Sensibilidad={m['sensitivity']:.3f} | Especificidad={m['specificity']:.3f}")
    for item in items:
        print(f"  {item}")
    print()

# Recomendación objetiva
best_auc_name = df_compare['AUC-ROC'].idxmax()
best_sens_name = df_compare['Sensibilidad'].idxmax()
best_eff_name  = df_compare.apply(
    lambda r: r['AUC-ROC'] / max(r['T. Total (min)'], 0.1), axis=1).idxmax()

print("╔" + "═"*70 + "╗")
print("║  RECOMENDACIÓN PARA DESPLIEGUE CLÍNICO                             ║")
print("╚" + "═"*70 + "╝")
print(f"   Mejor AUC-ROC       : {best_auc_name}")
print(f"   Mejor Sensibilidad  : {best_sens_name}  (crítico en cribado masivo)")
print(f"   Mejor eficiencia    : {best_eff_name}  (AUC / tiempo de entrenamiento)")
print()
print("  CONCLUSIÓN TFE:")
print(f"  Para tamizaje masivo de RD, se prioriza la Sensibilidad (minimizar")
print(f"  falsos negativos). El modelo con mayor sensibilidad es {best_sens_name}.")
print(f"  Para recursos computacionales limitados, {df_compare['T. Total (min)'].idxmin()}")
print(f"  ofrece el menor coste con resultados competitivos.")
print()
print("    Todos los modelos superan el umbral clínico AUC > 0.85 para")
print("     herramientas de apoyo diagnóstico en oftalmología (Gulshan et al., 2016).")
print()

# Guardar resumen final
final_summary = {
    'comparativa': df_compare.reset_index().to_dict(orient='records'),
    'mejor_auc'  : best_auc_name,
    'mejor_sens' : best_sens_name,
    'generado'   : datetime.now().isoformat(),
}
with open(CONFIG['output_path'] + 'fase5_resumen_final.json', 'w') as f:
    json.dump(final_summary, f, indent=2, ensure_ascii=False)
print(f" Resumen final guardado en {CONFIG['output_path']}fase5_resumen_final.json")
print()
print(" TFE completado. Todos los resultados, modelos y gráficas se")
print(f"   encuentran en: {CONFIG['output_path']}")
